In [55]:
import numpy as np
import os
import pandas as pd
import torch

In [56]:
df=pd.read_csv('./fashion-dataset/styles.csv', usecols=['id','articleType'])
df.head()

,id,articleType
0,15970,Shirts
1,39386,Jeans
2,59263,Watches
3,21379,Track Pants
4,53759,Tshirts


In [57]:
df.shape

(44446, 2)

In [58]:
df['articleType'].value_counts()

articleType
Tshirts                7070
Shirts                 3217
Casual Shoes           2846
Watches                2542
Sports Shoes           2036
                       ... 
Shoe Laces                1
Mens Grooming Kit         1
Body Wash and Scrub       1
Suits                     1
Ipad                      1
Name: count, Length: 143, dtype: int64

In [59]:
#apply data balacning, transform images , standardize images,labelencode the images

In [60]:
path_ref={}
for path in os.listdir('./fashion-dataset/images'):
    path_ref[path.split('.')[0]]=f'./fashion-dataset/images/{path}'

In [61]:
path_ref

{'9733': './fashion-dataset/images/9733.jpg',
 '14147': './fashion-dataset/images/14147.jpg',
 '52112': './fashion-dataset/images/52112.jpg',
 '6400': './fashion-dataset/images/6400.jpg',
 '34297': './fashion-dataset/images/34297.jpg',
 '24084': './fashion-dataset/images/24084.jpg',
 '12536': './fashion-dataset/images/12536.jpg',
 '54563': './fashion-dataset/images/54563.jpg',
 '15259': './fashion-dataset/images/15259.jpg',
 '35189': './fashion-dataset/images/35189.jpg',
 '44770': './fashion-dataset/images/44770.jpg',
 '4217': './fashion-dataset/images/4217.jpg',
 '47279': './fashion-dataset/images/47279.jpg',
 '36480': './fashion-dataset/images/36480.jpg',
 '16750': './fashion-dataset/images/16750.jpg',
 '50705': './fashion-dataset/images/50705.jpg',
 '26693': './fashion-dataset/images/26693.jpg',
 '37946': './fashion-dataset/images/37946.jpg',
 '3578': './fashion-dataset/images/3578.jpg',
 '40516': './fashion-dataset/images/40516.jpg',
 '10321': './fashion-dataset/images/10321.jpg',


In [62]:
df['id']=df['id'].astype(str)
df['id']=df['id'].map(path_ref)

In [63]:
df

,id,articleType
0,./fashion-dataset/images/15970.jpg,Shirts
1,./fashion-dataset/images/39386.jpg,Jeans
2,./fashion-dataset/images/59263.jpg,Watches
3,./fashion-dataset/images/21379.jpg,Track Pants
4,./fashion-dataset/images/53759.jpg,Tshirts
...,...,...
44441,./fashion-dataset/images/17036.jpg,Casual Shoes
44442,./fashion-dataset/images/6461.jpg,Flip Flops
44443,./fashion-dataset/images/18842.jpg,Tshirts
44444,./fashion-dataset/images/46694.jpg,Perfume and Body Mist


In [ ]:
from torchvision.transforms import transforms
train_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [65]:
from PIL import Image
from torch.utils.data import Dataset,DataLoader
class CustomData(Dataset):
    def __init__(self,paths,labels,transforms=None):
        self.paths=paths
        self.labels=torch.tensor(labels,dtype=torch.long)
        self.transforms=transforms
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, index):
        path=self.paths.iloc[index]
        img=Image.open(path).convert('RGB')
        img=self.transforms(img)
        return img,self.labels[index]

In [66]:
hold=df['articleType'].value_counts()
values_to_keep = hold[hold > 50].index
df = df[df['articleType'].isin(values_to_keep)]
df.dropna(inplace=True)

/var/folders/jh/0djpqpqx0zdgc726txgst9z40000gn/T/ipykernel_5203/569486955.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [67]:
from sklearn.model_selection import train_test_split
X_train,X_over,y_train,y_over=train_test_split(df['id'],df['articleType'],test_size=0.2,shuffle=True)
X_test,X_val,y_test,y_val=train_test_split(X_over,y_over,test_size=0.5,shuffle=True)

In [68]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_train=le.fit_transform(y_train)
y_val=le.transform(y_val)
y_test=le.transform(y_test)

In [69]:
np.unique(y_train)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69])

In [70]:
for i in np.unique(y_train):
    print(le.inverse_transform([i]))

['Accessory Gift Set']
['Backpacks']
['Bangle']
['Belts']
['Boxers']
['Bra']
['Bracelet']
['Briefs']
['Capris']
['Caps']
['Casual Shoes']
['Clutches']
['Cufflinks']
['Deodorant']
['Dresses']
['Duffel Bag']
['Dupatta']
['Earrings']
['Face Moisturisers']
['Flats']
['Flip Flops']
['Formal Shoes']
['Foundation and Primer']
['Fragrance Gift Set']
['Free Gifts']
['Handbags']
['Heels']
['Highlighter and Blush']
['Innerwear Vests']
['Jackets']
['Jeans']
['Jewellery Set']
['Kajal and Eyeliner']
['Kurta Sets']
['Kurtas']
['Kurtis']
['Laptop Bag']
['Leggings']
['Lip Gloss']
['Lipstick']
['Lounge Pants']
['Nail Polish']
['Necklace and Chains']
['Night suits']
['Nightdress']
['Pendant']
['Perfume and Body Mist']
['Ring']
['Sandals']
['Sarees']
['Scarves']
['Shirts']
['Shorts']
['Skirts']
['Socks']
['Sports Sandals']
['Sports Shoes']
['Stoles']
['Sunglasses']
['Sweaters']
['Sweatshirts']
['Ties']
['Tops']
['Track Pants']
['Trousers']
['Trunk']
['Tshirts']
['Tunics']
['Wallets']
['Watches']


In [71]:
train_dataset=CustomData(X_train,y_train,transforms=train_transforms)
test_dataset=CustomData(X_test,y_test,transforms=test_transforms)
val_dataset=CustomData(X_val,y_val,transforms=test_transforms)


In [72]:
train_dataset.__len__()

34680

In [73]:
train_dataloader=DataLoader(train_dataset,batch_size=32,pin_memory=True,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=32,pin_memory=True,shuffle=True)
val_dataloader=DataLoader(val_dataset,batch_size=32,pin_memory=True,shuffle=True)

In [74]:
len(np.unique(y_train))

70

In [75]:
import torch.nn as nn
class Model(nn.Module):
    def __init__(self,n_channels):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(n_channels,32,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(num_features=32),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(32,64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(num_features=64),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(64,128,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(num_features=128),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(128,128,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(num_features=128),
            nn.MaxPool2d(kernel_size=2,stride=2),
        )
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear(25088,out_features=128),
            nn.Dropout(),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.Dropout(),
            nn.ReLU(),
            nn.Linear(64,70)
        )
    def forward(self,x):
        out=self.features(x)
        return self.classifier(out)

In [76]:
device=torch.device('mps')
model=Model(3).to(device)
optimizer=torch.optim.Adam(params=model.parameters(),lr=0.0001)
from sklearn.utils.class_weight import compute_class_weight
class_weights=compute_class_weight(class_weight='balanced',classes=np.unique(y_train),y=y_train)
loss_fn=nn.CrossEntropyLoss(weight=torch.tensor(class_weights,dtype=torch.float).to(device))

In [77]:
# epochs=30
# from tqdm import tqdm
# for i in range(epochs):
#     tot_loss=0
#     correct1=0
#     total1=0
#     correct2=0
#     total2=0
#     for batch_features,batch_labels in tqdm(train_dataloader,desc='Training',leave=False):
#         batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
#         output=model(batch_features)
#         loss=loss_fn(output,batch_labels)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         tot_loss+=loss.item()
#         output=torch.argmax(output,dim=1)
#         correct1+=(output==batch_labels).sum().item()
#         total1+=len(batch_features)

#     model.eval()
#     for batch_features,batch_labels in tqdm(val_dataloader,desc='Validation',leave=False):
#         batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
#         output=model(batch_features)
#         output=torch.argmax(output,dim=1)
#         correct2+=(output==batch_labels).sum().item()
#         total2+=len(batch_features)
#     print(f'Epochs : {i+1}. Loss : {tot_loss/len(train_dataloader)} Training Accuracy :{correct1/total1} Validation Accuracy:{correct2/total2}')

In [78]:
# from my_model_module import Model  # Replace with actual path to your Model class
import pickle

model = pickle.load(open('./model.pkl', 'rb'))


In [79]:
# correct=0
# total=0
# model.eval()
# with torch.no_grad():
#     for batch_features,batch_labels in tqdm(test_dataloader,desc='Testing',leave=False):
#         batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)
        
#         output=model(batch_features)
#         output=torch.argmax(output,dim=1)
#         correct+=(output==batch_labels).sum().item()
#         total+=len(batch_features)
#     print(correct/total)

In [80]:
import pickle
# pickle.load(open('model.pkl','rb'))

In [81]:
model= pickle.load(open('model.pkl','rb'))

In [114]:
path='./test105.jpeg'
img=Image.open(path).convert('RGB')
transformed_img=train_transforms(img).to(device).view(1,3,224,224)
# print(transformed_img.shape)
output=model(transformed_img)
print(output)
output=torch.argmax(output,dim=1).to(device='cpu').numpy()
print(le.inverse_transform(output))


tensor([[-3.6700e+01, -9.3616e+00, -1.2760e+01, -7.5002e+00, -2.5769e+01,
         -2.8335e+01, -1.2913e+01, -2.6327e+01, -1.5425e+01, -1.0957e+01,
         -1.1356e+01, -1.6166e+01, -1.8516e+01, -1.0469e+01, -7.3181e+00,
         -1.8165e+01, -7.8963e+00, -1.0833e+01, -1.8054e+01, -1.2561e+01,
         -1.1509e+01, -1.7041e+01, -1.5246e+01, -1.5001e+01,  4.3698e+00,
         -8.6272e+00, -1.4554e+01, -2.1010e+01, -1.1015e+01, -5.6718e+00,
         -1.5808e+01, -1.8743e+01, -1.8016e+01, -1.0800e+01, -4.1076e+00,
         -2.2433e+01, -3.6776e+01, -9.5113e+00, -1.6532e+01, -1.4470e+01,
         -1.0386e+01, -7.4462e+00, -9.5170e+00, -1.6559e+01,  1.7543e-02,
         -8.2499e+00, -5.6121e+00, -1.4886e+01, -1.0640e+01, -9.2623e+00,
         -1.1672e+00, -6.1853e+00, -5.7890e+00, -1.4327e+01, -7.1048e-01,
         -2.9587e+01, -1.8012e+01, -3.4480e+00, -8.2117e+00, -1.9983e+01,
         -6.2600e+00, -5.5339e+00, -5.5994e+00, -9.6818e-01, -1.1332e+01,
         -3.0935e+01, -6.0512e+00, -8.

In [115]:
df['articleType'].value_counts()

articleType
Tshirts                  7069
Shirts                   3215
Casual Shoes             2846
Watches                  2542
Sports Shoes             2036
                         ... 
Lounge Pants               61
Jewellery Set              58
Fragrance Gift Set         57
Highlighter and Blush      53
Boxers                     52
Name: count, Length: 70, dtype: int64

In [120]:
df[df['articleType']=='Bra'].value_counts().sum()

np.int64(477)